# 08 — Modelos de Machine Learning

## Objetivo

Entrenar y comparar modelos con **validación cruzada anidada** (5 folds
externos × 3 internos). La selección de características y los hiperparámetros
se ajustan dentro de cada fold externo. Las predicciones out-of-fold (OOF) se
usan para la evaluación del notebook 09.

## 0. Configuración

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA, V2, REP = ROOT / "data", ROOT / "data" / "v2", ROOT / "reports"
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

## 1. Resultados de la CV anidada

In [ ]:
summary = pd.read_csv(V2 / "summary_models.csv")
summary.round(3)

## 2. Curvas ROC y Precision-Recall (LR)

In [ ]:
display(Image(filename=str(REP / "v2" / "roc_pr_lr.png")))

## 3. Matriz de confusión del modelo recomendado

In [ ]:
display(Image(filename=str(REP / "v2" / "confusion_lr_texture.png")))

## 4. Umbral clínico

In [ ]:
display(Image(filename=str(REP / "v2" / "umbral_lr_texture.png")))

## 5. Hiperparámetros elegidos por fold

In [ ]:
folds = pd.read_csv(V2 / "folds__texture__lr.csv")
folds[["fold", "balanced_accuracy", "sensitivity_tb", "specificity", "ap", "best_params"]]

## 6. Umbral anidado (sin optimismo)

In [ ]:
nested = pd.read_csv(V2 / "nested_threshold__texture__lr.csv")
nested[["fold", "threshold", "balanced_accuracy", "sensitivity_tb", "specificity", "ap"]].round(3)

El umbral se elige **dentro de cada fold de entrenamiento** con el objetivo
sensibilidad ≥ 0.90 y se aplica al fold externo. Así no se reutiliza el test
para elegir el umbral.

## 7. Modelo final

In [ ]:
manifest = json.loads((ROOT / "models" / "v2_lr_texture_manifest.json").read_text(encoding="utf-8"))
{k: manifest[k] for k in ["model", "feature_set", "n_features_input",
                          "n_features_selected", "best_params",
                          "prevalence", "threshold_youden", "threshold_sens90"]}

## Conclusiones

- LR sobre textura es el modelo recomendado: interpretable, barato y con
  probabilidades.
- XGBoost, SVM y RF se reportan como comparación; no se elige un modelo por
  una diferencia mínima sin significación.
- El umbral no se optimiza sobre las mismas predicciones OOF que se evalúan.

**Siguiente etapa:** `09_Evaluacion_Comparativa.ipynb`.